## SETTING UP

In [ ]:
# IMPORTS
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import neuropythy as ny
import k3d

libpath = '/mmfs1/home/achegu/repos/mesh-annot/src'
if libpath not in sys.path:
    sys.path.append(libpath)

import mesh_annot

In [ ]:
# CONFIGURE FILE PATHS
# Had to omit 601127 from validation because it some NaN values
# This is beyond my understanding because I specifically omit them while
# loading in the data. Also, it has never been an issue before this,
# so I don't know what that's about.
hcp_dataset_path = '/gscratch/scrubbed/achegu/data/mesh-annot/surface/'
save_path = '/gscratch/scrubbed/achegu/data/mesh-annot/'
partition_path = './partition.json'

with open(partition_path, 'r') as fl:
    import json
    partition = json.load(fl)

In [ ]:
# CREATE DATASETS
trn_sids = partition['trn']
val_sids = partition['val']

properties=('convexity', 'thickness', 'curvature', 'myelin', 'prf_y', 'prf_cod', 'prf_sigma')
target=('prf_x')

trn_dset = mesh_annot.HCPDataset(hcp_dataset_path, trn_sids, properties=properties, target=target)
val_dset = mesh_annot.HCPDataset(hcp_dataset_path, val_sids, properties=properties, target=target)

## TRAINING

In [ ]:
mesh_annot = mesh_annot.reload()

In [ ]:
# TRAIN THE MODEL
run = mesh_annot.train(
    trn_dset,
    val_dset,
    properties=properties,
    target=target,
    epochs=100,
    log=print,
    savedir=None
)

In [ ]:
# EXTRACT A MODEL PREDICTION
model = run.model
graph_data = val_dset[2].to(mesh_annot.config.device) # The [2] can be changed
all_predictions = run.model(graph_data)
prediction = all_predictions.detach().cpu().numpy()

In [ ]:
# VISUALIZE THE MODEL PREDICTION
subject = ny.freesurfer_subject('fsaverage')
lh_surface = subject.hemis['lh'].surfaces['white']

# plot = k3d.plot()
ny.cortex_plot(lh_surface, color=prediction[:,0], cmap='Blues', vmin=0, vmax=1)
# plot.display()

In [ ]:
# TESTING TO SEE WHICH COLUMNS HAVE NaNs
import torch
from torch_geometric.loader import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = mesh_annot.SplineGCN(in_channels=10, out_channels=1).to(device)
data = DataLoader(val_dset, batch_size=1, shuffle=False)
i=0

for batch in data:
    batch = batch.to(device)
    if torch.isnan(batch.x).any():
        # Check if a batch has NaNs
        print(f'--- Nan Detected in Batch---')
        print(f'Batch graph sizes: {batch.ptr}')

        # Find which nodes (rows) contain NaNs
        nan_nodes = torch.isnan(batch.x).any(dim=1)
        node_indices = torch.where(nan_nodes)[0]
        print(f'Node indices with NaNs: {node_indices.tolist()}')

        # Find which specific feature channels (columns) are causing the NaNs
        nan_features = torch.isnan(batch.x).any(dim=0)
        feature_indices = torch.where(nan_features)[0]
        print(f'Feature channels with NaNs: {feature_indices.tolist()}')
        
    print("All done", i)
    i += 1